# Hospital image vs training data — colour comparison

The new clinical slide at [data/raw/hospital/Image_02_series2.tif](../data/raw/hospital/Image_02_series2.tif) was scanned and stained at a different lab than the public NMSC dataset the segmentation models were trained on, so the colour distribution drifts. This notebook quantifies that drift across several views — direct RGB, HSV, and H&E colour-deconvolution — so the magnitude of the stain shift is concrete rather than just "the pink looks off".

Reference dataset is `nmsc-2x` (the normalized full-image dataset every pipeline pins as `val:` per [CLAUDE.md](../CLAUDE.md)).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import tifffile
from plotly.subplots import make_subplots
from skimage.color import rgb2hed, rgb2hsv

from config import paths
from pato.dataset import DatasetViewer
from pato.visualize import show_image, show_side_by_side

## Load the hospital image

It's a BigTIFF with LZW compression (~700 MB decoded), so we use `tifffile` + `imagecodecs` rather than Pillow. The full array stays in memory but most cells work off a coarse decimation for speed.

In [ ]:
hospital_path = paths.data_raw / "hospital" / "Image_02_series2.tif"
hospital = tifffile.imread(hospital_path, key=0)
print(f"hospital image: shape={hospital.shape} dtype={hospital.dtype} "
      f"size={hospital.nbytes / 1e6:.0f} MB")

In [ ]:
# 20x decimation → ~870 × 670; fine for a visual overview.
hospital_thumb = hospital[::20, ::20]
show_image(hospital_thumb)

## Load the training reference

`nmsc-2x` is the normalized full-image source every pipeline trains against (see CLAUDE.md). We use it directly — colour statistics on the source are what the network actually saw, regardless of which tile cache a particular run trained on.

In [ ]:
train_view = DatasetViewer(root=paths.data_processed / "nmsc-2x", split="train")
print(f"{len(train_view)} training samples")
train_view.sample_ids[:5]

In [ ]:
# Three training samples next to the hospital thumbnail, picked deterministically.
picks = [train_view[i].image for i in (0, 10, 25)]
show_side_by_side(
    hospital_thumb,
    *picks,
    titles=["hospital", *[f"train: {train_view.sample_ids[i]}" for i in (0, 10, 25)]],
    height=380,
)

## Drop background before comparing

Slide backgrounds are mostly empty glass — near-white pixels that have nothing to do with the stain. A naive mean over the whole image is dominated by how much glass each sample contains, not by stain. We compute a *tissue mask* (luminance < 220) once and only ever aggregate over those pixels.

220 is liberal — it keeps light pink/eosin regions while still cutting the bulk of the glass. Bumping it further has very little effect on the distributions below.

In [ ]:
def tissue_pixels(image: np.ndarray, lum_thresh: int = 220, max_pixels: int = 500_000) -> np.ndarray:
    """Return an (N, 3) uint8 array of tissue pixels from an RGB image.

    Filter on luminance, then random-subsample (deterministic seed) to cap memory
    so this works on the gigapixel hospital slide.
    """
    flat = image.reshape(-1, 3)
    lum = flat.mean(axis=1)
    tissue = flat[lum < lum_thresh]
    if len(tissue) > max_pixels:
        rng = np.random.default_rng(0)
        tissue = tissue[rng.choice(len(tissue), size=max_pixels, replace=False)]
    return tissue

In [ ]:
hospital_tissue = tissue_pixels(hospital)
print(f"hospital tissue pixels (subsampled): {len(hospital_tissue):,}")

# Pool tissue pixels across N training images so the comparison is to the
# *distribution* the model saw, not a single arbitrary slide.
N_TRAIN_SAMPLES = 30
rng = np.random.default_rng(0)
train_idx = rng.choice(len(train_view), size=min(N_TRAIN_SAMPLES, len(train_view)), replace=False)
train_tissue = np.concatenate(
    [tissue_pixels(train_view[int(i)].image, max_pixels=50_000) for i in train_idx]
)
print(f"training tissue pixels (pooled over {len(train_idx)} slides): {len(train_tissue):,}")

## Per-channel RGB histograms

Same tissue-only pixels, plotted as overlapping density histograms. A right-shifted hospital distribution = brighter overall; a left-shift in green relative to red/blue = stronger pink/magenta tilt; etc.

In [ ]:
def channel_hist(hospital_px: np.ndarray, train_px: np.ndarray, names=("R", "G", "B")) -> go.Figure:
    fig = make_subplots(rows=1, cols=3, subplot_titles=names, shared_yaxes=True)
    for i, name in enumerate(names):
        fig.add_trace(
            go.Histogram(x=hospital_px[:, i], name="hospital", marker_color="crimson",
                         opacity=0.55, histnorm="probability density", nbinsx=80,
                         legendgroup="hospital", showlegend=(i == 0)),
            row=1, col=i + 1,
        )
        fig.add_trace(
            go.Histogram(x=train_px[:, i], name="training", marker_color="steelblue",
                         opacity=0.55, histnorm="probability density", nbinsx=80,
                         legendgroup="training", showlegend=(i == 0)),
            row=1, col=i + 1,
        )
    fig.update_layout(barmode="overlay", height=320, margin=dict(l=40, r=10, t=40, b=30))
    return fig

channel_hist(hospital_tissue, train_tissue)

In [ ]:
def summary(px: np.ndarray) -> dict[str, np.ndarray]:
    return {"mean": px.mean(axis=0), "std": px.std(axis=0), "median": np.median(px, axis=0)}

for name, px in [("hospital", hospital_tissue), ("training", train_tissue)]:
    s = summary(px)
    print(f"{name:9s}  mean={s['mean'].round(1).tolist()}  "
          f"median={s['median'].astype(int).tolist()}  std={s['std'].round(1).tolist()}")

## Where does the hospital slide sit *inside* the training distribution?

Each training slide has its own mean RGB — there's already a spread across the dataset (different blocks, different scanners). To check whether the hospital slide is genuinely an outlier or just at the edge of the existing variation, we plot each training slide's tissue-mean as a point in R/B space and overlay the hospital mean.

Out-of-cloud = a real domain shift the network hasn't seen.

In [ ]:
per_slide_means = np.stack([
    tissue_pixels(train_view[i].image, max_pixels=20_000).mean(axis=0)
    for i in range(len(train_view))
])
hospital_mean = hospital_tissue.mean(axis=0)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=per_slide_means[:, 0], y=per_slide_means[:, 2],
    mode="markers", marker=dict(size=6, color="steelblue", opacity=0.6),
    name="training slides",
    hovertext=[train_view.sample_ids[i] for i in range(len(train_view))],
))
fig.add_trace(go.Scatter(
    x=[hospital_mean[0]], y=[hospital_mean[2]],
    mode="markers", marker=dict(size=14, color="crimson", symbol="x", line=dict(width=2)),
    name="hospital",
))
fig.update_layout(
    xaxis_title="mean R (tissue pixels)",
    yaxis_title="mean B (tissue pixels)",
    height=420, margin=dict(l=40, r=10, t=20, b=40),
)
fig

## HSV: hue and saturation drift

RGB conflates lightness with chromaticity. HSV separates them — *hue* tells us where in the colour wheel the stain sits (pink-vs-purple) and *saturation* tells us how vivid it is (washed-out vs strong). For H&E this is usually the most diagnostic view of stain drift.

In [ ]:
def to_hsv(px_u8: np.ndarray) -> np.ndarray:
    # rgb2hsv expects float [0, 1]; shape (N, 1, 3) keeps it as a 'tiny image'.
    return rgb2hsv((px_u8 / 255.0).reshape(-1, 1, 3)).reshape(-1, 3)

hospital_hsv = to_hsv(hospital_tissue)
train_hsv = to_hsv(train_tissue)

channel_hist(hospital_hsv, train_hsv, names=("hue", "saturation", "value"))

In [ ]:
for name, px in [("hospital", hospital_hsv), ("training", train_hsv)]:
    print(f"{name:9s}  hue median={np.median(px[:, 0]):.3f}  "
          f"sat median={np.median(px[:, 1]):.3f}  val median={np.median(px[:, 2]):.3f}")

## H&E colour deconvolution

Ruifrok & Johnston's deconvolution projects RGB onto the haematoxylin/eosin/DAB stain vectors. The H channel ≈ "nuclei density" and the E channel ≈ "cytoplasm/stroma intensity". Drift here is the most clinically meaningful version of the question — it says whether the *staining* itself moved, controlling for whatever colour balance the scanner applied on top.

In [ ]:
def to_hed(px_u8: np.ndarray) -> np.ndarray:
    return rgb2hed((px_u8 / 255.0).reshape(-1, 1, 3)).reshape(-1, 3)

hospital_hed = to_hed(hospital_tissue)
train_hed = to_hed(train_tissue)

channel_hist(hospital_hed, train_hed, names=("H (haematoxylin)", "E (eosin)", "D (residual)"))

In [ ]:
for name, px in [("hospital", hospital_hed), ("training", train_hed)]:
    print(f"{name:9s}  H median={np.median(px[:, 0]):.3f}  "
          f"E median={np.median(px[:, 1]):.3f}")

## Tile-level visual

The histograms tell us the distribution is shifted; this confirms it at the *appearance* level the model actually sees. We pick a tissue-dense 1024×1024 patch from the hospital slide and one from a training slide and look at them side-by-side.

In [ ]:
def find_tissue_tile(
    image: np.ndarray, size: int = 1024, lum_thresh: int = 220, min_frac: float = 0.5
) -> np.ndarray | None:
    """Scan a stride-`size//2` grid and return the densest tile, or None if all are mostly glass.

    Deterministic. Tile clips to image size if the slide is smaller than `size`.
    """
    H, W, _ = image.shape
    size = min(size, H, W)
    best, best_frac = None, 0.0
    for y in range(0, H - size + 1, max(size // 2, 1)):
        for x in range(0, W - size + 1, max(size // 2, 1)):
            tile = image[y:y + size, x:x + size]
            frac = float((tile.mean(axis=2) < lum_thresh).mean())
            if frac > best_frac:
                best, best_frac = tile, frac
    return best if best_frac >= min_frac else None

hospital_tile = find_tissue_tile(hospital)
# Try training slides in order until one yields a tissue-dense tile.
train_tile, picked_id = None, None
for i in range(len(train_view)):
    tile = find_tissue_tile(train_view[i].image)
    if tile is not None:
        train_tile, picked_id = tile, train_view.sample_ids[i]
        break
print(f"training tile from: {picked_id}")
show_side_by_side(
    hospital_tile, train_tile,
    titles=["hospital tile", f"training tile ({picked_id})"], height=500,
)

## Reading these results

What to look for, in increasing order of clinical relevance:

1. **RGB means / scatter plot** — does the hospital cross land outside the cloud of training slides? If yes, this is genuinely out-of-distribution colour, not just within-variation.
2. **HSV** — a hue shift toward purple (lower hue values) or saturation collapse points at a stain-protocol or scanner-balance difference, not just brightness.
3. **HED** — a shift in the H channel = different haematoxylin intensity; in E = different eosin. This is what stain-normalization methods (Macenko, Reinhard, Vahadane) target.

If the gap is large in (3), the next step is to either:
- run training-time stain augmentation (`monai.transforms` doesn't ship this — `torchstain` or `staintools` do); or
- normalize hospital slides toward a fixed training-slide reference at inference time.